###Ingest Biller_master parquet file   
1.read file using dataframe reader api   
2.add metadata columns - source file,ingestion timestamp     
3.write bronze tables using dataframe write api

In [0]:
dbutils.widgets.text("p_batch_date","")
v_batch_date = dbutils.widgets.get("p_batch_date")

In [0]:
%run ../00-common/01.environment_config


In [0]:
%run ../00-common/02.bronze_functions

In [0]:
source_file = f"{raw_path}/{v_batch_date}/biller_master/"

In [0]:
table_name = f"{catalog_name}.{bronze_schema}.biller_master"

In [0]:
from pyspark.sql.types import *

In [0]:
billers_schema = StructType([
  StructField('BillerID', StringType()),
  StructField('BillerName', StringType()),
  StructField('BillerCategory', StringType()),
  StructField('Status', StringType())
])

In [0]:
billers_df = spark.read.format('parquet') \
                .option('mode', 'FAILFAST') \
                .load(source_file)

In [0]:
billers_audit = add_ingestion_metadata(billers_df)

In [0]:
billers_final= billers_audit.withColumn("batch_date", F.lit(v_batch_date))

In [0]:
billers_final.write.mode('overwrite').partitionBy('batch_date').option('replaceWhere',f"batch_date = '{v_batch_date}'").saveAsTable(table_name)

In [0]:
%sql
SELECT * FROM payment_app.bronze.biller_master


BillerID,BillerName,BillerCategory,Status,CreatedDate,ingestion_timestamp,source_file,batch_date
DTH01,Digital TV Services,Entertainment,Active,2026-08-13T10:33:48.286Z,2026-08-22T13:10:33.094Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/biller_master/bbps.BILLER_MASTER.parquet,2026-08-13
GAS01,State Gas Corporation,Gas,Active,2026-08-13T10:33:48.286Z,2026-08-22T13:10:33.094Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/biller_master/bbps.BILLER_MASTER.parquet,2026-08-13
KSEB,Kerala State Electricity Board,Electricity,Active,2026-08-13T10:33:48.286Z,2026-08-22T13:10:33.094Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/biller_master/bbps.BILLER_MASTER.parquet,2026-08-13
TEL01,National Telecom Services,Telecom,Active,2026-08-13T10:33:48.286Z,2026-08-22T13:10:33.094Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/biller_master/bbps.BILLER_MASTER.parquet,2026-08-13
WTR01,City Water Supply Board,Water,Active,2026-08-13T10:33:48.286Z,2026-08-22T13:10:33.094Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/biller_master/bbps.BILLER_MASTER.parquet,2026-08-13
